# Agrupamento de Notícias e Análise Semântica com LLM (Local vs Nuvem)

Este Jupyter Notebook foi desenvolvido como material de suporte acadêmico (seminário/projeto) sob orientação do **Prof. José Alfredo F. Costa (UFRN)**.

O objetivo é executar e documentar de forma reprodutível um bloco de experimentos de Processamento de Linguagem Natural (PLN) e Aprendizado de Máquina Supervisionado/Não-Supervisionado sobre uma base de dados de notícias expandidas (6 categorias).

---

## 1. Configuração e Importação de Bibliotecas

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from dotenv import load_dotenv

# Scikit-Learn e ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Embeddings locais
from sentence_transformers import SentenceTransformer
import umap

warnings.filterwarnings('ignore')
load_dotenv('backend/.env')
print("Bibliotecas importadas com sucesso!")

## 2. Carga e Limpeza de Dados

Lemos o arquivo `Base_dados_textos_6_classes.csv` e removemos valores nulos (`NaN`) para garantir que os algoritmos de vetorização funcionem corretamente.

In [ ]:
csv_path = "Base_dados_textos_6_classes.csv"
df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
print(f"Tamanho original: {len(df)} registros.")

# Limpeza de nulos nas colunas de texto
df = df.dropna(subset=["Texto Original", "Texto Expandido"]).reset_index(drop=True)
print(f"Tamanho após limpeza: {len(df)} registros.")

# Visualização da distribuição das classes reais
print("\nDistribuição por Categoria Real:")
print(df["Categoria"].value_counts())
df.head(3)

## 3. Extração de Atributos (TF-IDF vs Embeddings Densos)

Comparamos duas formas de representação textual:

1. **Representação Clássica Frequencial (TF-IDF):**
   $$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$

2. **Representação Semântica Densa (Dense Embeddings):** Vetores densos de 384 dimensões gerados por rede neural pré-treinada localmente (`all-MiniLM-L6-v2`).

In [ ]:
texts = df["Texto Expandido"].tolist()
true_labels = df["Classe"].values

# 3.1. TF-IDF
pt_stopwords = ['a','de','o','que','e','em','um','para','com','se','no','na','por','mais','ao','as','dos','das','como','mas']
vectorizer = TfidfVectorizer(max_features=1000, stop_words=pt_stopwords)
X_tfidf = vectorizer.fit_transform(texts).toarray()
print(f"Matriz TF-IDF gerada: {X_tfidf.shape}")

# 3.2. SentenceTransformers (Dense)
st_model = SentenceTransformer('all-MiniLM-L6-v2')
X_dense = st_model.encode(texts, show_progress_bar=True)
print(f"Embeddings Densos gerados: {X_dense.shape}")

## 4. Agrupamento e Métricas Científicas

Para medir a qualidade do agrupamento de forma formal e acadêmica, calculamos:

*   **Métricas Não Supervisionadas (Estrutura geométrica):**
    *   **Coeficiente de Silhueta ($s$):** Mede a coesão intra-grupo contra a separação extra-grupo. Varia em $[-1, +1]$.
        $$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
        onde $a(i)$ é a distância média intra-grupo e $b(i)$ é a distância média para o vizinho mais próximo.

*   **Métricas Supervisionadas (Concordância com as classes originais):**
    *   **Adjusted Rand Index (ARI):** Mede o grau de concordância das partições corrigindo pelo acaso. Varia em $[-1, 1]$. O valor $1.0$ representa acerto perfeito.
    *   **Normalized Mutual Information (NMI):** Alinhamento com base em teoria da informação. Varia em $[0, 1]$.

In [ ]:
def evaluate_clustering(X, true_labels, pred_labels):
    n_clusters = len(set(pred_labels) - {-1})
    if n_clusters > 1 and n_clusters < len(pred_labels):
        sil = silhouette_score(X, pred_labels)
        db = davies_bouldin_score(X, pred_labels)
    else:
        sil, db = -1.0, -1.0
    
    ari = adjusted_rand_score(true_labels, pred_labels)
    nmi = normalized_mutual_info_score(true_labels, pred_labels)
    
    return {"K": n_clusters, "Silhueta": sil, "Davies-Bouldin": db, "ARI": ari, "NMI": nmi}

# Experimentos com K-Means (K=6) nas duas matrizes
kmeans_tfidf = KMeans(n_clusters=6, random_state=42, n_init=10)
labels_tfidf = kmeans_tfidf.fit_predict(X_tfidf)

kmeans_dense = KMeans(n_clusters=6, random_state=42, n_init=10)
labels_dense = kmeans_dense.fit_predict(X_dense)

res_tfidf = evaluate_clustering(X_tfidf, true_labels, labels_tfidf)
res_dense = evaluate_clustering(X_dense, true_labels, labels_dense)

print("Métricas de Comparação (K-Means K=6):")
df_metrics = pd.DataFrame([res_tfidf, res_dense], index=["TF-IDF", "Dense Embeddings"])
df_metrics

## 5. Visualização 2D com UMAP

Usamos o UMAP (*Uniform Manifold Approximation and Projection*) para projetar os vetores de alta dimensionalidade em duas coordenadas, permitindo comparar visualmente a qualidade do agrupamento semântico contra a representação TF-IDF.

In [ ]:
# Reduções UMAP
reducer_tfidf = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
coords_tfidf = reducer_tfidf.fit_transform(X_tfidf)

reducer_dense = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
coords_dense = reducer_dense.fit_transform(X_dense)

# Plotando a comparação
fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=150)
cmap = plt.get_cmap('tab10')

# TF-IDF Clusters
axes[0, 0].scatter(coords_tfidf[:, 0], coords_tfidf[:, 1], c=labels_tfidf, cmap=cmap, s=35, alpha=0.8)
axes[0, 0].set_title("TF-IDF: Clusters Preditos (K-Means)")
axes[0, 0].grid(True, linestyle='--', alpha=0.3)

# TF-IDF Classes Reais
axes[0, 1].scatter(coords_tfidf[:, 0], coords_tfidf[:, 1], c=true_labels, cmap=cmap, s=35, alpha=0.8)
axes[0, 1].set_title("TF-IDF: Classes Originais (Ground Truth)")
axes[0, 1].grid(True, linestyle='--', alpha=0.3)

# Dense Clusters
axes[1, 0].scatter(coords_dense[:, 0], coords_dense[:, 1], c=labels_dense, cmap=cmap, s=35, alpha=0.8)
axes[1, 0].set_title("Dense Embeddings: Clusters Preditos (K-Means)")
axes[1, 0].grid(True, linestyle='--', alpha=0.3)

# Dense Classes Reais
axes[1, 1].scatter(coords_dense[:, 0], coords_dense[:, 1], c=true_labels, cmap=cmap, s=35, alpha=0.8)
axes[1, 1].set_title("Dense Embeddings: Classes Originais (Ground Truth)")
axes[1, 1].grid(True, linestyle='--', alpha=0.3)

plt.suptitle("Projeção Bidimensional UMAP (Comparativo TF-IDF vs Dense)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Seleção de Textos Críticos (Medoides vs Fronteiras)

Utilizamos o valor de silhueta individual de cada amostra para encontrar:
1.  **Medoides (Centrais):** Elementos com maior silhueta no grupo (ou menor distância geométrica ao centroide).
2.  **Fronteiras (Ambíguos):** Elementos com menor silhueta (próximos a 0 ou negativos), representando textos com intersecções temáticas (ex. notícia de tecnologia com teor econômico).

In [ ]:
# Cálculo do coeficiente de silhueta para cada elemento nos embeddings densos
sil_samples = silhouette_samples(X_dense, labels_dense)

# Seleção de amostras do Cluster 0
cluster_target = 0
cluster_indices = np.where(labels_dense == cluster_target)[0]

# Amostras com maior Silhueta (Centrais)
sorted_core = np.argsort(sil_samples[cluster_indices])[::-1]
core_indices = [cluster_indices[i] for i in sorted_core[:3]]

# Amostras com menor Silhueta (Fronteiras/Ambíguos)
sorted_border = np.argsort(sil_samples[cluster_indices])
border_indices = [cluster_indices[i] for i in sorted_border[:3]]

print(f"=== ANÁLISE DE LANDMARKS - CLUSTER #{cluster_target} ===\n")

print("--- TEXTOS CENTRAIS (NÚCLEO DO TEMA) ---")
for idx in core_indices:
    print(f" - ID: {idx} | Silhueta: {sil_samples[idx]:.3f}")
    print(f"   Título: {df.iloc[idx]['Texto Original']}")
    print(f"   Categoria Real: {df.iloc[idx]['Categoria']}\n")
    
print("--- TEXTOS DE FRONTEIRA (AMBÍGUOS / HÍBRIDOS) ---")
for idx in border_indices:
    print(f" - ID: {idx} | Silhueta: {sil_samples[idx]:.3f}")
    print(f"   Título: {df.iloc[idx]['Texto Original']}")
    print(f"   Categoria Real: {df.iloc[idx]['Categoria']}\n")